# Image processing notebook: From overlap corrected to transmission 

### 00 - exp 1XX acquisition 00

##  Initial settings

### Import libraries
Import all the required libraries

In [1]:
import sys
sys.path.append(r'..\01_Functions')
from step_functions import *
from dict_functions import *
from proc_functions import *
from img_functions import *
%matplotlib inline

### Select directories
Select the source directory. This directory is where the images **after** the overlap correction were saved.
Select the destination directory. Here is where the transmission images are going to be saved.

In [2]:
# %load select_directory('src_dir')
src_dir = r"J:\900 Varia\2021\000_tony_data\03_Processed_step_by_step\old_testing\testing_overlap_corrected_sa\exp1XX"

In [3]:
# %load select_directory('dst_dir')
dst_dir = r"J:\900 Varia\2021\000_tony_data\03_Processed_step_by_step\01_New_transmission_results\exp1XX_full"

### Check folders to process

In [4]:
stack_dict = prep_stack_dict(src_dir)
for key in stack_dict.keys():
    print(key)

00_ob
01_so_ref
02_sa
02_sa_2_2
03_ob_end


## Process the reference folder (if not done before)
Or in case it is not previously saved.

In case the parameters and ref_dict are stored in the global variables, you can load them with : `%store -r "name given"`

It is **important** to not process the images fully before having played with them to find the right parameters (there is a notebook called playgound todo this)

In [5]:
#%store -r exp_param

In [6]:
ref_folder = ['01_so_ref']

In [7]:
ref_seq =  [stack_averaging,
            outlier_removal, # it will remove the 0's in the image, if any
            scrubbing_correction_dict, # it will take all the OB frames and not just HE and LE
            SBKG_correction_dict, 
            TFC_correction_dict]

In [8]:
ref_param = {}
BB_mask = get_img(src_dir + '/bb_mask_ref_full.fits')
nca = [408, 399, 43, 10]

add_to_dict(ref_param, ['threshold', 'BB_mask', 'nca', 'use_ref'], [0, BB_mask, nca, False])

In [9]:
ref_dict = full_processing (src_dir, dst_dir, proc_folder = ref_folder, sequence = ref_seq, 
                 proc_parameters = ref_param, img_name = 'full_intensity', save = True)

Reading Images: 100%|████████████████████████████| 3/3 [00:21<00:00,  7.17s/it]


Experiment 01_so_ref in process...


Processing SBKG Correction: 100%|████████████████| 1/1 [00:18<00:00, 18.65s/it]


Using proc_dict as both img and ref to calculate TFC


Processing TFC Correction: 100%|█████████████████| 1/1 [00:00<00:00,  4.50it/s]


Saving images as a single acquisition


Writing Images: 100%|████████████████████████████| 1/1 [00:21<00:00, 21.05s/it]

Total time: 997s


## Experiment Avg  Processing

In [10]:
exp_param = {}
BB_mask = get_img(src_dir + '/bb_mask_full.fits')
nca = [408, 399, 43, 10]

disp = float(-0.1999999)
M = np.array([[1,0,disp], [0,1,0], [0,0,1]])

add_to_dict(exp_param, ['threshold', 'BB_mask', 'nca', 'use_ref', 'M'], [0, BB_mask, nca, False, M])
add_to_dict(exp_param, ['ref_dict'], [ref_dict])

In [14]:
exp_seq =  [stack_averaging,
            outlier_removal, # it will remove the 0's in the image, if any
            scrubbing_correction_dict, # it will take all the OB frames and not just HE and LE
            SBKG_correction_dict,
            image_registration_dict,
            TFC_correction_dict]

In [15]:
proc_folder = ['02_sa']

disp = float(-0.1999999)
M = np.array([[1,0,disp], [0,1,0], [0,0,1]])

exp_dict = full_processing (src_dir, dst_dir, proc_folder = proc_folder, sequence = exp_seq, 
                 proc_parameters = exp_param, img_name = 'avg_intensity', save = True)

Reading Images: 100%|████████████████████████████| 3/3 [00:22<00:00,  7.66s/it]


Experiment 02_sa in process...


Processing TFC Correction: 100%|█████████████████| 1/1 [00:00<00:00,  4.91it/s]


Saving images as a single acquisition


Writing Images: 100%|████████████████████████████| 1/1 [00:20<00:00, 20.95s/it]

Total time: 1071s


## Experiment HE_LE  Processing

In [16]:
HE_LE = (True, [15,30],[30,50])

add_to_dict(exp_param, ['HE_LE', 'save_energies'], [HE_LE, True])

In [17]:
dst_dir_new = r"J:\900 Varia\2021\000_tony_data\03_Processed_step_by_step\01_New_transmission_results\exp1XX_HE_LE"
exp_dict = full_processing (src_dir, dst_dir_new, proc_folder = proc_folder, sequence = exp_seq, 
                 proc_parameters = exp_param, img_name = 'HE_LE_intensity', save = True)

Reading Images: 100%|████████████████████████████| 3/3 [00:19<00:00,  6.58s/it]


Experiment 02_sa in process...


Processing TFC Correction: 100%|█████████████████| 1/1 [00:00<00:00,  4.92it/s]


Saving images as HE and LE stacks


Writing HE and LE Images: 100%|██████████████████| 1/1 [00:00<00:00,  2.26it/s]

Total time: 937s


## Experiment full HE_LE Processing

In [5]:
ref_folder = ['01_so_ref']

In [6]:
ref_seq =  [stack_averaging, #averages all acquisitions 
            outlier_removal, # it will remove the 0's in the image, if any
            scrubbing_correction_dict, # it will take all the OB frames and not just HE and LE
            binning_frames, # it will average frames to have a HE and a LE image because of the HE_LE parameter
            SBKG_correction_dict, 
            TFC_correction_dict]

In [7]:
ref_param = {}
BB_mask = get_img(src_dir + '/bb_mask_ref_full.fits')
nca = [408, 399, 43, 10]
HE_LE = (True, [15,30],[30,50])

add_to_dict(ref_param, ['threshold', 'BB_mask', 'nca', 'use_ref', 'HE_LE'], [0, BB_mask, nca, False, HE_LE])
add_to_dict(ref_param, ['save_energies'], [ True])

In [8]:
ref_dict = full_processing (src_dir, dst_dir, proc_folder = ref_folder, sequence = ref_seq, 
                 proc_parameters = ref_param, img_name = 'so_ref_energies', save = True)

Reading Images: 100%|████████████████████████████| 3/3 [00:21<00:00,  7.01s/it]


Experiment 01_so_ref in process...


Processing SBKG Correction: 100%|████████████████| 1/1 [00:00<00:00,  2.37it/s]


Using proc_dict as both img and ref to calculate TFC


Processing TFC Correction: 100%|████████████████| 1/1 [00:00<00:00, 124.67it/s]


Saving images as HE and LE stacks


Writing HE and LE Images: 100%|██████████████████| 1/1 [00:00<00:00,  2.51it/s]

Total time: 953s


In [9]:
exp_param = {}
BB_mask = get_img(src_dir + '/bb_mask_full.fits')
nca = [408, 399, 43, 10]
HE_LE = (True, [15,30],[30,50])

disp = float(-0.1999999)
M = np.array([[1,0,disp], [0,1,0], [0,0,1]])

add_to_dict(exp_param, ['threshold', 'BB_mask', 'nca', 'use_ref', 'M'], [0, BB_mask, nca, False, M])
add_to_dict(exp_param, ['ref_dict', 'save_energies', 'HE_LE'], [ref_dict, True, HE_LE])

In [10]:
exp_seq =  [outlier_removal, # it will remove the 0's in the image, if any
            scrubbing_correction_dict,
            binning_frames,# it will take all the OB frames and not just HE and LE
            SBKG_correction_dict,
            image_registration_dict,
            TFC_correction_dict]

In [11]:
proc_folder = ['02_sa']

exp_dict = full_processing (src_dir, dst_dir, proc_folder = proc_folder, sequence = exp_seq, 
                 proc_parameters = exp_param, img_name = 'HE_LE_energies_intensity', save = True)

Reading Images: 100%|████████████████████████████| 3/3 [00:18<00:00,  6.13s/it]


Experiment 02_sa in process...


Processing TFC Correction: 100%|██████████████| 24/24 [00:00<00:00, 195.03it/s]


Saving images as HE and LE stacks


Writing HE and LE Images: 100%|████████████████| 24/24 [00:09<00:00,  2.57it/s]

Total time: 1147s
